# 02｜單車租借月報：三階提問實作

[![在 Colab 開啟](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roberthsu2003/PythonForDataAnalysis/blob/master/%E7%9C%8B%E6%87%82%E8%B3%87%E6%96%99%E8%88%87%E9%81%B8%E5%B0%8D%E5%9C%96%E8%A1%A8/%E5%92%8CAI%E5%8D%94%E4%BD%9C%E7%9A%84%E4%B8%89%E9%9A%8E%E6%8F%90%E5%95%8F%E6%B3%95/02_%E5%96%AE%E8%BB%8A%E7%A7%9F%E5%80%9F%E6%9C%88%E5%A0%B1.ipynb)

> **趨勢 × 折線圖**｜課堂合成資料；一列代表 **2025 年的一個月份**。

| ① 看懂資料 | ② 一題一圖 | ③ 三輪追問 |
| :--- | :--- | :--- |
| 欄位與邊界 | 數字表與圖 | 核對數字 → 挑戰推論 → 修訂結論 |

**你負責提問與驗收，AI 負責寫程式。** 先執行第 0 格，依序將 prompt 貼給 Colab Gemini 或能讀取筆記本的 AI；參考程式供核對。第二階完成後，留在同一段對話，接著問第三階的三輪。

**核心問題：**2025 年每月租借次數如何變化？哪個月最高？　｜　[下載資料集](./資料集/%E5%96%AE%E8%BB%8A%E7%A7%9F%E5%80%9F%E6%9C%88%E5%A0%B1.xlsx)

In [ ]:
# 0｜載入 xlsx，並沿用前面章節的繁體中文字型設定
from pathlib import Path
from urllib.request import urlretrieve
import glob
import os
import pandas as pd
import matplotlib.pyplot as plt

filename = '單車租借月報.xlsx'
file_path = next((p for p in [Path('資料集') / filename, Path(filename), Path('和AI協作的三階提問法/資料集') / filename] if p.exists()), None)
if file_path is None:
    file_path = Path(filename)
    urlretrieve('https://raw.githubusercontent.com/roberthsu2003/PythonForDataAnalysis/master/%E7%9C%8B%E6%87%82%E8%B3%87%E6%96%99%E8%88%87%E9%81%B8%E5%B0%8D%E5%9C%96%E8%A1%A8/%E5%92%8CAI%E5%8D%94%E4%BD%9C%E7%9A%84%E4%B8%89%E9%9A%8E%E6%8F%90%E5%95%8F%E6%B3%95/%E8%B3%87%E6%96%99%E9%9B%86/%E5%96%AE%E8%BB%8A%E7%A7%9F%E5%80%9F%E6%9C%88%E5%A0%B1.xlsx', file_path)
%pip install -q mplfonts
import mplfonts
from matplotlib import font_manager
from mplfonts import use_font
for font_file in glob.glob(os.path.join(os.path.dirname(mplfonts.__file__), 'fonts', '*.[ot]tf')):
    font_manager.fontManager.addfont(font_file)
use_font('Noto Sans CJK SC')
plt.rcParams['axes.unicode_minus'] = False
df = pd.read_excel(file_path, sheet_name='資料')
print('資料檔：', file_path, '；列數：', len(df))
display(df.head())

---

## ① 第一階｜看懂資料

**先自己判斷：**月份＝時間條件；租借次數、雨天數＝指標。檢查月份是否連續、是否重複或缺月。

**複製給 AI**

```text
請讀取已載入的 df（單車租借月報.xlsx／工作表「資料」）。顯示前 5 列、總列數、欄位名稱、每欄缺值數；說明一列代表什麼，將欄位分成指標、條件、標識。檢查資料的時間範圍、重複或缺值，提出 2 個可以回答、1 個不能回答的問題並說明理由。先不要畫圖，也不要下結論。
```

**驗收：**筆數、缺值與欄位角色，都要和 xlsx 一致。

In [ ]:
print('資料列數：', len(df))
print('欄位：', list(df.columns))
display(df.head(5))
display(df.isna().sum().rename('缺值數').to_frame())

---

## ② 第二階｜一個問題、一張圖

**複製給同一段 AI 對話**

```text
延續第一階，回答「2025 年每月租借次數如何變化？哪個月最高？」；這是趨勢問題。先按月份由早到晚排序，顯示月份與租借次數的數字表。只畫一張折線圖（Line Chart）：X 軸＝月份，Y 軸＝租借次數（次），每月有資料點，標出最高月份與數值。不要把雨天數畫在同一個 Y 軸。標題、坐標軸用繁體中文，沿用前面章節的 mplfonts／Noto Sans CJK SC 字型設定。不要畫其他圖。
```

**驗收：**先核對數字表，再核對圖表類型、坐標軸、單位與中文字。

In [ ]:
ordered = df.sort_values('月份').copy()
ordered['月份標籤'] = ordered['月份'].dt.strftime('%Y-%m')
display(ordered[['月份標籤', '租借次數']])
peak = ordered.loc[ordered['租借次數'].idxmax()]
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(ordered['月份標籤'], ordered['租借次數'], marker='o', linewidth=2.5, color='#277DA1')
ax.scatter(peak['月份'].strftime('%Y-%m'), peak['租借次數'], s=120, color='#E07A5F', zorder=3)
ax.annotate(f"最高：{int(peak['租借次數']):,} 次", (peak['月份'].strftime('%Y-%m'), peak['租借次數']), xytext=(8, 12), textcoords='offset points')
ax.set(xlabel='月份', ylabel='租借次數（次）', title='2025 年每月單車租借次數')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

---

## ③ 第三階｜接著追問三輪

沿用第二階的數字表與圖，不要重開對話。

### 第 1 輪｜核對數字與排除資料

```text
核對折線圖 12 個點與數字表的順序和數值；列出最高、最低月份與次數，核對最高點標註。檢查缺月、重複或缺值；有問題就先修正。
```

In [ ]:
expected = pd.date_range('2025-01-01', '2025-12-01', freq='MS')
assert ordered['月份'].is_unique and list(ordered['月份']) == list(expected)
assert ordered['租借次數'].notna().all()
low = ordered.loc[ordered['租借次數'].idxmin()]
print('月份數：', len(ordered))
print('最高：', peak['月份'].strftime('%Y-%m'), int(peak['租借次數']))
print('最低：', low['月份'].strftime('%Y-%m'), int(low['租借次數']))

### 第 2 輪｜挑戰推論

```text
從這 12 個月的月總次數，能否判定雨天變多造成租借減少，或說全年持續成長？指出可觀察的變化和檢驗原因所需的其他資料。不要重畫圖。
```

### 第 3 輪｜修訂三句結論

```text
用三句繁體中文寫結論：①最高月份與租借次數；②資料涵蓋 2025 年 12 個月的月總次數，沒有逐日資料；③這張圖不能證明雨天造成變化。數字以核對後的表格為準。
```

> ✍️ **我的驗收紀錄：**哪個數字被核對？哪句推論被改寫？（雙擊此格記下來。）

---

[返回第 7 課](./README.md)　｜　[返回學習地圖](../README.md)